# Explaining an image–text score with SigLIP 2
Run in a fresh CPU runtime, separately from the language notebook. We compare
two descriptions, inspect gradients and Integrated Gradients, then mask equal
areas and examine neighbours in a small, explicitly supplied comparison pool.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

PINS = {'transformers':'5.22.1','captum':'0.9.0','matplotlib':'3.11.2',
        'scikit-image':'0.26.0','psutil':'7.2.2'}
# Retain Colab's preloaded plotting library.
missing=[]
for package,version in PINS.items():
    try:
        if metadata.version(package)!=version and package!='matplotlib':missing.append(f'{package}=={version}')
    except metadata.PackageNotFoundError:
        missing.append(f'{package}=={version}')
if importlib.util.find_spec('torch') is None:missing.append('torch==2.15.0')
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])

## Loading the photographs and the model
These three scikit-image photographs are CC0: Chelsea by Stefan van der Walt,
coffee by Rachel Michetti, and camera by Lav Varshney. We display the exact
resized pixels seen by the model when placing attributions on a photograph.

In [ ]:
import os,json,time,hashlib
from pathlib import Path
import numpy as np
from PIL import Image,ImageFilter
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
from captum.attr import Saliency,IntegratedGradients
from captum.influence import SimilarityInfluence
from transformers import AutoModel,AutoProcessor
from skimage import data
import psutil

os.environ['TOKENIZERS_PARALLELISM']='false'
torch.manual_seed(17);torch.set_num_threads(min(4,os.cpu_count() or 1))
DEVICE=torch.device('cpu')
OUT=Path(os.environ.get('CH17_VISION_OUT','outputs/vision'));OUT.mkdir(parents=True,exist_ok=True)
STARTED=time.perf_counter()
MODEL_ID='google/siglip2-base-patch16-224'
MODEL_REVISION='75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2'
processor=AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
model=AutoModel.from_pretrained(MODEL_ID,revision=MODEL_REVISION,dtype=torch.float32,
                              attn_implementation='eager').to(DEVICE).eval()
model.requires_grad_(False)
photos={name:Image.fromarray(get()).convert('RGB') for name,get in
        [('chelsea',data.chelsea),('coffee',data.coffee),('camera',data.camera)]}
photo_hashes={name:hashlib.sha256(np.asarray(im).tobytes()).hexdigest() for name,im in photos.items()}
pixels=processor(images=photos['chelsea'],return_tensors='pt').pixel_values.to(DEVICE)
mean=torch.tensor(processor.image_processor.image_mean).view(1,3,1,1)
std=torch.tensor(processor.image_processor.image_std).view(1,3,1,1)
shown=(pixels.cpu()*std+mean)[0].permute(1,2,0).numpy().clip(0,1)
print(f'Input shape: {tuple(pixels.shape)}; parameters: {sum(p.numel() for p in model.parameters()):,}')

def show_pixels(values):
    return (values.detach().cpu() * std + mean)[0].permute(1, 2, 0).numpy().clip(0, 1)

def save_reader_figure(fig, stem):
    fig.savefig(OUT / f'{stem}.png', dpi=300, facecolor='white')
    fig.savefig(OUT / f'{stem}.svg', facecolor='white')
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(5.4, 2.05), layout='constrained')
for ax, view, label in zip(
    axes, (photos['chelsea'], shown),
    ('Original photograph', 'Model input: 224 × 224'),
):
    ax.imshow(view)
    ax.set_title(label, fontsize=9)
    ax.axis('off')
fig.suptitle('From the photograph to the model input', fontsize=11, fontweight='bold')
save_reader_figure(fig, 'chelsea_model_input')

## Defining the image score
We hold the descriptions fixed and subtract their cosine similarities. SigLIP's
built-in scaled logits and sigmoid scores are not used here. This is a chosen
contrast, not a calibrated probability or a detector of whether any cat exists.

In [ ]:
TEXTS=['a photo of a cat','a photo of a dog']
def pooled(value):
    return value.pooler_output if hasattr(value,'pooler_output') else value
with torch.no_grad():
    tokens=processor(text=TEXTS,padding='max_length',return_tensors='pt').to(DEVICE)
    text_features=F.normalize(pooled(model.get_text_features(**tokens)),dim=-1).detach()

def image_features(values):
    return F.normalize(pooled(model.get_image_features(pixel_values=values)),dim=-1)

def contrast(values):
    similarities=image_features(values)@text_features.T
    return similarities[:,0]-similarities[:,1]

with torch.no_grad():
    original_score=contrast(pixels).item()
    original_similarities=(image_features(pixels)@text_features.T)[0].tolist()
print(f'Cat similarity: {original_similarities[0]:.4f}; dog similarity: {original_similarities[1]:.4f}')
print(f'Cat-minus-dog score: {original_score:+.4f}')

## Computing gradients and choosing reference photographs
Normalized zero represents middle grey with this processor. It does not mean
zero light or “no influence”. A black reference is -1, because mean and standard
deviation are both 0.5. A blurred photograph asks a third, different question.

In [ ]:
assert torch.allclose(mean,torch.full_like(mean,.5)) and torch.allclose(std,torch.full_like(std,.5))
grey=torch.zeros_like(pixels)
blurred_image=Image.fromarray((shown*255).round().astype('uint8')).filter(ImageFilter.GaussianBlur(4))
blurred=processor(images=blurred_image,return_tensors='pt').pixel_values.to(DEVICE)
ig_started=time.perf_counter()
gradients=Saliency(contrast).attribute(pixels.clone().requires_grad_(),abs=False).detach()
ig=IntegratedGradients(contrast)
attribution,delta=ig.attribute(pixels,baselines=grey,n_steps=256,internal_batch_size=1,
                             return_convergence_delta=True)
blur_attribution,blur_delta=ig.attribute(pixels,baselines=blurred,n_steps=64,internal_batch_size=1,
                                       return_convergence_delta=True)
with torch.no_grad():
    grey_score=contrast(grey).item();blur_score=contrast(blurred).item()
ig_error=float(delta.item());blur_error=float(blur_delta.item())
print(f'Grey reference: {grey_score:+.4f}; IG sum: {attribution.sum().item():+.4f}; residual: {ig_error:+.6f}')
print(f'Blur reference: {blur_score:+.4f}; IG sum: {blur_attribution.sum().item():+.4f}; residual: {blur_error:+.6f}')
assert abs(ig_error)<max(.002,.03*abs(original_score-grey_score))
assert abs(blur_error)<max(.002,.03*abs(original_score-blur_score))
assert torch.isfinite(attribution).all() and torch.isfinite(gradients).all()
ig_seconds=time.perf_counter()-ig_started

fig, axes = plt.subplots(1, 3, figsize=(5.4, 1.85), layout='constrained')
for ax, values, label, score in zip(
    axes, (grey, blurred, pixels), ('Grey baseline', 'Blurred baseline', 'Original input'),
    (grey_score, blur_score, original_score),
):
    ax.imshow(show_pixels(values))
    ax.set_title(f'{label}\nScore {score:+.4f}', fontsize=9)
    ax.axis('off')
fig.suptitle('Two starting images, one photograph to explain', fontsize=11, fontweight='bold')
save_reader_figure(fig, 'vision_baselines')

In [ ]:
PATCH=32
GRID=224//PATCH
signed_pixels=attribution.sum(dim=1)[0].detach().cpu().numpy()
def region_sums(values):
    return values.reshape(GRID,PATCH,GRID,PATCH).sum(axis=(1,3))
patch_scores=region_sums(signed_pixels)
gradient_regions=region_sums(gradients.square().sum(dim=1).sqrt()[0].cpu().numpy())
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9.5})
fig=plt.figure(figsize=(5.4,3.0))
grid=fig.add_gridspec(2,3,height_ratios=[1,.07],left=.03,right=.98,
                     top=.78,bottom=.24,wspace=.09,hspace=.16)
axes=[fig.add_subplot(grid[0,j]) for j in range(3)]
for ax in axes:ax.imshow(shown);ax.axis('off')
axes[0].set_title('Model input',fontweight='bold')
g=axes[1].imshow(gradient_regions,extent=(0,224,224,0),cmap='magma',alpha=.7,
                 vmin=0,vmax=gradient_regions.max(),interpolation='nearest')
axes[1].set_title('Gradient\nmagnitude',fontweight='bold')
limit=max(float(np.abs(patch_scores).max()),1e-8)
a=axes[2].imshow(patch_scores,extent=(0,224,224,0),cmap='RdBu_r',alpha=.7,
                 vmin=-limit,vmax=limit,interpolation='nearest')
axes[2].set_title('Integrated\nGradients',fontweight='bold')
fig.colorbar(g,cax=fig.add_subplot(grid[1,1]),orientation='horizontal',label='Gradient magnitude')
fig.colorbar(a,cax=fig.add_subplot(grid[1,2]),orientation='horizontal',label='Signed contribution')
fig.suptitle('Choose the score before reading a heatmap',fontweight='bold',fontsize=11)
fig.savefig(OUT/'vision_attribution.png',dpi=260);fig.savefig(OUT/'vision_attribution.svg');plt.show()

## Optional: starting from black instead of grey
Run the preceding cells, then set `TRY_BLACK_BASELINE` to `True` and run this cell.
Chelsea’s photograph and the original grey results stay unchanged.

In [ ]:
TRY_BLACK_BASELINE = False
if TRY_BLACK_BASELINE:
    black = torch.full_like(pixels, -1)
    black_attribution, black_delta = ig.attribute(
        pixels, baselines=black, n_steps=256, internal_batch_size=1,
        return_convergence_delta=True,
    )
    with torch.no_grad():
        black_score = contrast(black).item()
    black_error = float(black_delta.item())
    print(f'Black reference: {black_score:+.4f}; '
          f'IG sum: {black_attribution.sum().item():+.4f}; residual: {black_error:+.6f}')
    assert torch.isfinite(black_attribution).all()
    assert abs(black_error) < max(.002, .03 * abs(original_score - black_score))
    black_regions = region_sums(black_attribution.sum(dim=1)[0].detach().cpu().numpy())
    # Use one colour scale so the panels can be compared directly.
    limit = max(float(np.abs(patch_scores).max()), float(np.abs(black_regions).max()), 1e-8)
    fig, axes = plt.subplots(1, 2, figsize=(5.4, 2.8), layout='constrained')
    for ax, label, values in zip(axes, ('Grey baseline', 'Black baseline'), (patch_scores, black_regions)):
        ax.imshow(shown)
        heat = ax.imshow(values, extent=(0, 224, 224, 0), cmap='RdBu_r',
                         alpha=.7, vmin=-limit, vmax=limit, interpolation='nearest')
        ax.set_title(label)
        ax.axis('off')
    fig.colorbar(heat, ax=axes, orientation='horizontal', label='Signed contribution')
    plt.show()

## Testing equal-area masks
Rank non-overlapping 32×32 regions by their signed IG sum. Mask the top four
with the same grey used as the reference, then compare with twenty seeded
random masks of four regions each. These edited images may be unfamiliar to the
model; a bigger score drop is local intervention evidence, not a universal test.

In [ ]:
BUDGET=4
chosen=np.argsort(patch_scores.ravel())[-BUDGET:]
def masked_input(indices):
    values=pixels.clone()
    for index in indices:
        row,col=divmod(int(index),GRID)
        values[:,:,row*PATCH:(row+1)*PATCH,col*PATCH:(col+1)*PATCH]=0
    return values
rng=np.random.default_rng(17)
random_indices=[rng.choice(GRID*GRID,BUDGET,replace=False) for _ in range(20)]
with torch.no_grad():
    chosen_score=contrast(masked_input(chosen)).item()
    random_scores=[contrast(masked_input(indices)).item() for indices in random_indices]
mask_area=BUDGET/(GRID*GRID)
print(f'Masked area: {mask_area:.1%}; original: {original_score:+.4f}; chosen: {chosen_score:+.4f}')
print(f'Random scores: mean {np.mean(random_scores):+.4f}, range [{min(random_scores):+.4f}, {max(random_scores):+.4f}]')
fig=plt.figure(figsize=(5.4,3.0))
axes=[fig.add_axes(rect) for rect in [[.015,.22,.23,.57],[.26,.22,.23,.57],[.685,.24,.295,.55]]]
for ax,inds,title in zip(axes[:2],[chosen,random_indices[0]],['IG-selected\nregions','One random\ncontrol']):
    view=(masked_input(inds).cpu()*std+mean)[0].permute(1,2,0).numpy().clip(0,1)
    ax.imshow(view);ax.axis('off');ax.set_title(title,fontweight='bold',fontsize=9)
random_marks=axes[2].scatter(np.ones(20),random_scores,s=16,color='#738394',alpha=.7,label='Random controls')
selected_mark=axes[2].scatter([0],[chosen_score],color='#147d77',s=45,zorder=3,label='IG-selected mask')
original_line=axes[2].axhline(original_score,color='#555',linestyle='--',label='Original score')
axes[2].set_xticks([0,1],['IG-selected','Random'],fontsize=8)
axes[2].set_xlim(-.3,1.3)
axes[2].set_ylabel('Cat-minus-dog score',fontsize=8.5)
axes[2].tick_params(axis='y',labelsize=8)
axes[2].spines[['top','right']].set_visible(False)
fig.suptitle('Compare masks that cover the same area',fontweight='bold',fontsize=10.5,y=.98)
fig.legend([selected_mark,random_marks,original_line],['IG-selected mask','Random controls','Original score'],
           loc='lower center',bbox_to_anchor=(.5,.025),ncol=3,frameon=False,fontsize=7.5,
           handlelength=1.6,columnspacing=1.0)
fig.savefig(OUT/'mask_controls.png',dpi=260);fig.savefig(OUT/'mask_controls.svg');plt.show()

## Trying other descriptions and photographs
A positive cat-minus-dog margin on a coffee photograph would not imply a cat is
present. We also score coffee against a relevant alternative description.

In [ ]:
other_results={}
for name,photo in photos.items():
    values=processor(images=photo,return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        other_results[name]=contrast(values).item()
    print(f'{name}: cat-minus-dog {other_results[name]:+.4f}')
with torch.no_grad():
    alternative_tokens=processor(text=['a photo of a kitten','a photo of a puppy','a photo of a cup of coffee'],
                                 padding='max_length',return_tensors='pt').to(DEVICE)
    alternative_features=F.normalize(pooled(model.get_text_features(**alternative_tokens)),dim=-1)
    reworded=(image_features(pixels)@alternative_features[:2].T)[0]
    reworded_score=(reworded[0]-reworded[1]).item()
    coffee_pixels=processor(images=photos['coffee'],return_tensors='pt').pixel_values.to(DEVICE)
    coffee_feature=image_features(coffee_pixels)
    coffee_cat=(coffee_feature@text_features[0]).item()
    coffee_coffee=(coffee_feature@alternative_features[2]).item()
print(f'Kitten-minus-puppy on the same cat: {reworded_score:+.4f}')
print(f'Coffee photograph: cat similarity {coffee_cat:.4f}, coffee similarity {coffee_coffee:.4f}')

coffee_dog = (coffee_feature @ text_features[1]).item()
np.testing.assert_allclose(coffee_cat - coffee_dog, other_results['coffee'], atol=1e-6)
fig, (photo_ax, score_ax) = plt.subplots(
    1, 2, figsize=(5.4, 2.15), gridspec_kw={'width_ratios': [1, 1.8]}, layout='constrained',
)
photo_ax.imshow(show_pixels(coffee_pixels))
photo_ax.axis('off')
photo_ax.set_title('Coffee photograph', fontsize=9)
labels = ['Cat', 'Dog', 'Cup of coffee']
scores = [coffee_cat, coffee_dog, coffee_coffee]
bars = score_ax.barh(labels, scores, color=['#497190', '#74808a', '#146a5d'], height=.58)
score_ax.invert_yaxis()
score_ax.bar_label(bars, fmt='%.4f', fontsize=8.5, padding=4)
score_ax.set_xlim(0, max(scores) * 1.35)
score_ax.set_xlabel('Cosine similarity', fontsize=9)
score_ax.tick_params(axis='both', labelsize=9)
score_ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('The answer depends on the descriptions we offer', fontsize=10.5, fontweight='bold')
save_reader_figure(fig, 'coffee_description_scores')

## Finding neighbours in a supplied pool
Captum's class is named SimilarityInfluence, but this exercise retrieves nearby
representations among our three photographs. They are not asserted to be the
pretraining examples that caused this model's prediction.

In [ ]:
class ComparisonEncoder(nn.Module):
    def __init__(self,base):
        super().__init__();self.base=base;self.representation=nn.Identity()
    def forward(self,values):
        features=F.normalize(pooled(self.base.get_image_features(pixel_values=values)),dim=-1)
        return self.representation(features)

pool_names=list(photos)
pool_pixels=processor(images=list(photos.values()),return_tensors='pt').pixel_values.to(DEVICE)
pool=TensorDataset(pool_pixels)
comparison=ComparisonEncoder(model).eval()
def representation_cosine(query_features, source_features):
    # Captum may wrap a single queried layer in a singleton list.
    if isinstance(query_features, (list, tuple)):
        assert len(query_features) == 1
        query_features = query_features[0]
    return F.normalize(query_features.flatten(1), dim=1) @ F.normalize(
        source_features.flatten(1), dim=1).T

finder=SimilarityInfluence(comparison,'representation',pool,str(OUT/'neighbour_cache'),
                           model_id=MODEL_REVISION,batch_size=1,
                           similarity_metric=representation_cosine)
query=pixels.flip(-1)
with torch.no_grad():
    neighbours=finder.influence(query,top_k=3,load_src_from_disk=False)['representation']
    neighbour_ids,neighbour_similarities=neighbours
    expected_similarities=comparison(query)@comparison(pool_pixels).T
    order=expected_similarities.argsort(dim=-1,descending=True)
    torch.testing.assert_close(neighbour_ids,order)
    torch.testing.assert_close(neighbour_similarities,expected_similarities.gather(1,order))
ranked=[{'name':pool_names[i],'similarity':float(s)} for i,s in
        zip(neighbour_ids[0].tolist(),neighbour_similarities[0].tolist())]
print('Nearest supplied photographs:',ranked)
fig,axes=plt.subplots(1,4,figsize=(5.4,2.6),layout='constrained')
axes[0].imshow(np.fliplr(shown));axes[0].set_title('Flipped cat\n(query)',fontsize=9,fontweight='bold')
for ax,item in zip(axes[1:],ranked):
    ax.imshow(photos[item['name']]);ax.set_title(f"{item['name']}\ncosine {item['similarity']:.3f}",fontsize=9)
for ax in axes:ax.axis('off')
fig.suptitle('Neighbours come from the pool we provide',fontweight='bold',fontsize=11)
fig.savefig(OUT/'representation_neighbours.png',dpi=260);fig.savefig(OUT/'representation_neighbours.svg');plt.show()

## Saving the evidence
The convergence residual checks numerical integration, not the truth of an
explanation. We retain negative results and the full random-control scores.

In [ ]:
try:
    import resource
    peak=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_bytes=int(peak if sys.platform=='darwin' else peak*1024)
except ImportError:
    peak_bytes=getattr(psutil.Process().memory_info(),'peak_wset',None)
results={'model':MODEL_ID,'model_revision':MODEL_REVISION,'device':str(DEVICE),'dtype':'float32',
         'parameters':sum(p.numel() for p in model.parameters()),'descriptions':TEXTS,
         'photo_hashes':photo_hashes,'original_similarities':original_similarities,'original_score':original_score,
         'grey_score':grey_score,'blur_score':blur_score,'ig_steps':256,'ig_sum':attribution.sum().item(),
         'ig_delta':ig_error,'blur_ig_steps':64,'blur_ig_sum':blur_attribution.sum().item(),'blur_ig_delta':blur_error,
         'ig_seconds':ig_seconds,'patch_size':PATCH,'mask_budget':BUDGET,'mask_area_fraction':mask_area,
         'chosen_regions':chosen.tolist(),'chosen_score':chosen_score,'random_indices':[x.tolist() for x in random_indices],
         'random_scores':random_scores,'other_images':other_results,'reworded_score':reworded_score,
         'coffee_cat_similarity':coffee_cat,'coffee_coffee_similarity':coffee_coffee,'neighbours':ranked,
         'process_peak_rss_bytes':peak_bytes,'seconds_including_loading':time.perf_counter()-STARTED,
         'versions':{name:metadata.version(name) for name in ['torch',*PINS]}}
(OUT/'results.json').write_text(json.dumps(results,indent=2)+'\n')
np.savez(OUT/'attribution_arrays.npz',input=pixels.cpu().numpy(),gradients=gradients.cpu().numpy(),
         ig=attribution.detach().cpu().numpy(),ig_blur=blur_attribution.detach().cpu().numpy(),
         patch_scores=patch_scores)
print(f'Saved results to {OUT}; process peak memory bytes: {peak_bytes}')